### Baseline Experiment

#### This experiment was directly recreated using the information from the paper 'Flood susceptibility mapping in the Nyabarongo Catchment, Rwanda, based on data analysis and modeling'

In [ ]:
# This cell imports the tools needed for analysis.
# NumPy handles numeric data, Pandas handles tables, and GeoPandas works with spatial datasets such as shapefiles.
# Matplotlib and Seaborn are used to make plots and charts.
# scikit-learn contains the machine learning model and evaluation tools.
# Please bear in mind that these comments are a combination of my own knowledge + what was already in the notebook.

## Importing the necessary packages. 
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import geopandas as gpd 
import sklearn
import matplotlib.pyplot as plt
import shap # used for model explainability and feature importance.
import pickle # Pickle allows Python objects to be saved to disk and reused later.
import geocube # geocube is used to convert vector data into a raster grid for mapping.
import traceback #  a standard Python module for showing detailed error stacks.

from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, classification_report, cohen_kappa_score, confusion_matrix

In [ ]:
# This cell loads the shapefile that contains the flood sample locations.
# A shapefile is a spatial GIS (Geographic Information System) file with geometry and attribute data.
# The dataset contains flood sample points and their conditioning features.
# We display the first rows to confirm the data was read correctly.

# Read the shapefile.
# The shapefile includes 302 locations and were obtained from the flood inventory and data collections (coordinates of flood hit-locations).
# Flooded = 0 indicates non-flooded location while Flooded = 1 indicates flooded location. Label will be used as the target variable for the machine learning model as we are trying to predict the probability of flooding at a given location.
# Every location has values for the 10 flood conditioning features.
df = gpd.read_file(r"C:\Users\Sandhra George\FSM-data\Sample_points\Sample_points\Sample_points.shp")
df.head()

In [ ]:
# This cell prints a summary of the dataset.
# It helps us understand the range, average, and spread of each variable.

print(df.describe().T)

In [ ]:
# This cell checks for missing values.
# Missing values can affect model performance, so we need to know whether cleaning is needed.

## Check if there is null values
print(df.isnull().sum())
#df = df.dropna()

In [ ]:
# This cell renames the target column from 'Flooded' to 'Label'. ( I think the 'Label' column was originally named 'Flooded' in the shapefile.)
# This makes the target variable easier to understand in the model.
# It also shows the data types to confirm the columns are correct.

df = df.rename(columns={'Flooded':'Label'})
print(df.dtypes)

In [ ]:
# This cell shows how many sample points are flooded and how many are not.
# This helps check whether the dataset is balanced.

# Therefore, here we are trying to understand the data.
# We can see from the graph that the dataset includes the same number of flooded and not flooded locations.
sns.countplot(x="Label", data=df) # 0 - Not Flooded,    1 - Flooded

In [ ]:
# This cell separates the non-flooded and flooded rows into two groups.
# We remove the geometry and label columns so we can compare only the feature values between the two groups. Feature values are the input variables that will be used to predict flooding.

zero_labels = df.loc[df['Label'] == 0].drop(columns=['geometry', 'Label','x','y'])
ones_labels = df.loc[df['Label'] == 1].drop(columns=['geometry', 'Label','x','y'])

In [ ]:
# This cell displays the non-flooded subset of the data.

zero_labels

In [ ]:
# This cell displays the flooded subset of the data.
# Comparing both groups helps us understand how flood conditions differ.

ones_labels

In [ ]:
# This cell creates a correlation heatmap for non-flooded locations.
# Correlation shows how strongly features are related to one another.

# Correlation between the input features for non-flooded locations is shown.

corrMatrix = zero_labels.corr()
fig, ax = plt.subplots(figsize=(10,10)) # Sample figsize in inches
#sns.heatmap(df.iloc[:, 1:6:], annot=True, linewidths=.5, ax=ax)
sns.heatmap(corrMatrix, annot=True, linewidths=.5, ax=ax)

In [ ]:
# This cell creates a correlation heatmap for flooded locations.
# Comparing this with the previous heatmap helps reveal differences in flood drivers.

# Correlation between the input features for flooded locations is shown.

corrMatrix = ones_labels.corr()
fig, ax = plt.subplots(figsize=(10,10))         # Sample figsize in inches
#sns.heatmap(df.iloc[:, 1:6:], annot=True, linewidths=.5, ax=ax)
sns.heatmap(corrMatrix, annot=True, linewidths=.5, ax=ax)

In [ ]:
# This cell defines the target variable Y.
# Y contains the label values: 0 = not flooded, 1 = flooded.

# Define the dependent variable that needs to be predicted ('Label', whether the location is flooded or not).
Y = df["Label"].values
print(f"Dependent variable (Y) shape: {Y.shape}")
print(Y)

In [ ]:
# This cell defines the input features X. We will use these features to predict the target variable Y (whether a location is flooded or not).
# These are the flood-conditioning variables used to predict the class label.
# We remove the target and geometry columns because they are not input features.

X = df.drop(labels = ["Label",'geometry','x','y'], axis=1) 
features_list = list(X.columns)  # List features so we can rank them later
print(f"The list of features is: {features_list}")
print(f"Independent variables (X) shape: {X.shape}")
print(X)

In [ ]:
# This cell splits the data into training and test sets.
# The test set is kept unseen so we can evaluate the model fairly.

# Split data into train, validation and test to verify accuracy after fitting the model 
# Firstly split the data into train_validation and test datasets then split the train_validation dataset into train and validation datasets.
# The training dataset is used to train the model, the validation dataset is used for hyperparamter tuning and the testing dataset is used to test the model. 
# It is recommended to test the model with data that the model hasn't seen in the training process.
# Currently splitting the data into 80% training and 20% testing datasets. The training dataset will be further split into training and validation datasets in the next cell.

X_train_val, X_test, y_train_val, y_test = train_test_split(X, Y, test_size=0.20,shuffle=True, random_state=42)

In [ ]:
# This cell splits the remaining training data into training and validation sets.
# Validation helps choose model settings before final testing.
# Here we are splitting the training dataset into training and validation datasets. The training dataset will be used to train the model and the validation dataset will be used to tune the hyperparameters of the model. The validation dataset is used to evaluate the model performance during the training process.
# The training dataset is 64% of the total dataset and the validation dataset is 16% of the total dataset. The test dataset is 20% of the total dataset. 

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20,shuffle=True, random_state=42)

In [ ]:
# This cell defines the candidate settings for the Random Forest model.
# We test different values to find a combination that performs well.

# Define a parameter grid for Random Forest.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [ ]:
# This cell creates the Random Forest classifier object (an instance of the machine learning model we are using).

# Instantiate the Random Forest classifier.
model = RandomForestClassifier(random_state = 42)

In [ ]:
# This cell runs a randomised search for the best model parameters.
# Cross-validation checks which settings generalise best. Generalisation refers to how well the model performs on unseen data, which is crucial for real-world applications.

# Perform RandomizedSearchCV with cross-validation.
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid,
                                   n_iter=10, scoring='f1', cv=5, verbose=1, random_state=42)
random_search.fit(X_train, y_train)

In [ ]:
# This cell trains the Random Forest model on the training data.

# Train the model on training data.
model.fit(X_train, y_train)

In [ ]:
# This cell prints the best hyperparameters selected by the search.

# Get the best hyperparameters found.
best_model = random_search.best_estimator_
print("Best hyperparameters found: ", random_search.best_params_)

In [ ]:
# This cell makes flood predictions on the test dataset.

# Make predictions on the test set.
prediction = best_model.predict(X_test)

In [ ]:
# This cell displays the predictions generated by the model.

# Prediction are 1 (Flooded) and 0 (Not flooded).
prediction 

In [ ]:
# This cell plots the ROC curve and computes the AUC.
# ROC shows how well the model separates flooded and non-flooded cases.

# Draw the Receiver Operating Characteristics and estimate the Area under the curve (AUC).
# Assuming model is your classifier and X_test, y_test are your test data.
y_score = best_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)
print(f"AUC: {roc_auc:.15f}")

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# This cell asks the model for the probability of flooding for each test sample.

# In order to map the flood susceptibility, we need to predict the flood susceptibility (probability between 0 and 1)
prediction_prob=best_model.predict_proba(X_test)

In [ ]:
# This cell shows the probability output for every test case.
# Each row contains two values: probability of not flooded and probability of flooded.

# The probability of being not flooded and flooded for every location (Their summation equals 1).
prediction_prob

In [ ]:
# This cell extracts only the probability of the flooded class.
# This value is usually used for flood susceptibility mapping.

ls=prediction_prob[:,1]
ls.shape
print(ls) # Here I'm just printing the actual values.

In [ ]:
# This cell prints a classification report.
# It summarises precision, recall, and F1-score for each class. These are additional performance indices that are useful for evaluating classification models. They are important because they provide a more complete picture of model performance, especially in imbalanced datasets.

# As flood susceptibility is classification problem (flooded or not flooded), we can calculate some additional performance indices.
target_names=["Not Flooded","Flooded"]
print(classification_report(y_test, prediction, target_names=target_names))

In [ ]:
# This cell calculates Cohen's Kappa score.
# Kappa measures agreement while correcting for agreement that could happen by chance.


# Cohen's Kappa is a statistic that measures inter-rater agreement for categorical items. It is generally thought to be a more robust measure than simple percent agreement calculation, as κ takes into account the agreement occurring by chance.
print(cohen_kappa_score(y_test, prediction))

In [ ]:
# This cell computes the confusion matrix.
# The matrix shows how many actual flooded/non-flooded points were correctly or wrongly predicted.

cm = confusion_matrix(y_test, prediction)
labels = ["Not Flooded (0)", "Flooded (1)"]

cm_df = pd.DataFrame(
    cm,
    index=["Actual: Not Flooded", "Actual: Flooded"],
    columns=["Predicted: Not Flooded", "Predicted: Flooded"]
)

print(cm_df)

In [ ]:
# This cell calculates feature importance.
# The model tells us which variables contribute most to predicting flooding. Higher the value, more important the feature is in predicting flooding. This is useful for understanding which factors are most influential in flood susceptibility.

feature_imp = pd.Series(best_model.feature_importances_, index=features_list).sort_values(ascending=False)
print(f"Feature importances:\n{feature_imp}")

In [ ]:
# This cell saves the feature importance chart as an image file.

# Assuming 'feature_imp' is a DataFrame or Series containing feature importances.
ax = feature_imp.plot.barh()
fig = ax.get_figure()
fig.savefig("feature_imp.jpg")

In [ ]:
# This cell imports SHAP for model explanation.
# SHAP helps explain which features drove the model's prediction. It stands for SHapley Additive exPlanations and is based on game theory. It provides insight into how each feature contributes to the model's output.

shap.initjs()

In [ ]:
# This cell creates the SHAP explainer for the trained model.
# It also creates a waterfall plot for the first prediction.
# It shows which features pushed the prediction toward flooded or not flooded.

# (same syntax works for LightGBM, CatBoost, scikit-learn, transformers, Spark, etc.)
explainer = shap.Explainer(best_model)
shap_values = explainer(X_test)

# visualise the first prediction's explanation
shap.plots.waterfall(shap_values[0, : , 0])

In [ ]:
# This cell creates a beeswarm plot for all feature effects.
# It shows the overall influence of each feature across the dataset.
shap.plots.beeswarm(shap_values[:,:,0])

In [ ]:
# Visualise the first prediction's explanation with a force plot. This is a more interactive way to see how each feature contributes to the prediction for a specific instance. The force plot shows the base value (average model output) and how each feature pushes the prediction higher or lower.
shap.plots.force(shap_values[:,:,0])

#### Applying the model to the Nyabarongo Catchment

In [ ]:
# Step 1: Load the study-area data
# This file contains the wider catchment area for the flood susceptibility model.
# We are reading the shapefile that represents the whole Nyabarongo study area.
# This dataset is larger than the sample points and will be used to generate a flood-risk map across the region.

# Step 2: Save the study-area data in a Python-friendly format
# This keeps the spatial data in the same structure used by GeoPandas.
# This step prepares the data so we can work with it easily in Python before prediction.
gdf = gpd.read_file(r"C:\Users\Sandhra George\FSM-data\StudyArea\StudyArea\Nyabarongo_StudyNew.shp")

In [ ]:
# Step 3: Save the data to a .pkl file
# A .pkl file stores the dataset on disk so it can be loaded again later.
# This saves time because we do not need to read the shapefile again every time.
with open('converted_data.pkl', 'wb') as f:
    pickle.dump(gdf, f)

In [ ]:
# This cell displays the first rows of the study-area data.
gdf.head()

In [ ]:
# This cell removes the geometry column before prediction.
# The model only needs the feature values, not the spatial geometry itself.
X_hotspot0= gdf.drop(labels = ["geometry"], axis=1) 
X_hotspot0.head()

In [ ]:
# This cell keeps only the same feature columns that were used during model training.
# The order of the columns is important because the model expects the same input structure.
# If the data is not in the same order, the model may not interpret the features correctly.
cols = X.columns
cols

In [ ]:
# This cell selects only the feature columns used during model training.
# The order of these columns must match the training data exactly.
# If the order changes, the model may make incorrect predictions.
X_hotspot0 = X_hotspot0[cols]

# Display the first few rows to confirm the data is aligned correctly.
X_hotspot0.head()

In [ ]:
# This cell removes rows with missing values.
# The model cannot process incomplete data reliably.
print(X_hotspot0.isnull().sum())

In [ ]:
# This cell creates a clean version of the study-area data for prediction.
# We remove rows with missing values because the model cannot process incomplete inputs.
# `X_hotspot0` contains the feature values that will be used for prediction.
# `df` is the GeoDataFrame that will later receive the flood susceptibility score.
X_hotspot0 = X_hotspot0.dropna()
df = gdf.dropna()

In [ ]:
len(df)

In [ ]:
# This cell predicts the flood probability for every location in the study area.
# Higher values mean greater flood susceptibility.

prediction_prob=best_model.predict_proba(X_hotspot0)

In [ ]:
# This cell displays the model output probabilities.

prediction_prob

In [ ]:
# This cell extracts only the probability of the flooded class.
# This probability will be stored as the final flood susceptibility score.

ls_hotspot0=prediction_prob[:,1]
ls_hotspot0

In [ ]:
# This cell adds the flood susceptibility score to the study-area dataset.
# The new column is named FSM and stores the risk score.

df['FSM']=ls_hotspot0
df.head()

In [ ]:
try:
    from geocube.api.core import make_geocube
    print("make_geocube import OK")
except Exception:
    traceback.print_exc()

In [ ]:
# This cell creates a raster grid from the study-area features and flood susceptibility values.
# Raster grids are useful for generating flood risk maps.

geo_grid = make_geocube(
    vector_data=df,
    measurements=['FSM'],
    resolution=(-50, 50) # I have changed the resolution to (-50, 50) because the original resolution of (-20, 20) was too fine and caused memory issues. The new resolution is coarser but should still provide useful spatial detail. Can change it back when required.
)

In [ ]:
# This cell plots the flood susceptibility raster.
# This gives a visual map of flood risk across the study area.

geo_grid.FSM.plot()

In [ ]:
# This cell saves the raster as a GeoTIFF file.
# The output can be reused in GIS software or reports.

geo_grid.FSM.rio.to_raster("FSM.tif")